<a href="https://colab.research.google.com/github/HERO-DS/git-practice/blob/main/BeutifulSoup_Quote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# SECTION 1: INITIAL SETUP & SETUP EXTERNAL DEPENDENCIES
# ==============================================================================

# Import the HTTP library to download web pages from the internet
import requests

# Import the parsing library to read and search through HTML structures
from bs4 import BeautifulSoup

# Import a utility to write tabular data into comma-separated value files (unused in game)
from csv import writer

# Import a timing function to pause script execution and prevent server overload
from time import sleep

# Import a random selection function to pull a single random element from a list
from random import choice

# Initialize an empty list to store all scraped quote objects as dictionaries
all_quotes = []

# Define the immutable homepage URL of the target website
base_url = "http://quotes.toscrape.com/"

# Initialize the dynamic sub-path variable starting with the first page
url = "/page/1"




In [2]:
# ==============================================================================
# SECTION 2: THE DATA SCRAPING & PAGINATION ENGINE (WHILE LOOP BLOCK)
# ==============================================================================

# Loop continuously as long as the 'url' variable contains a valid string path
while url:

    # Combine the base URL and sub-path string, then execute an HTTP GET request to download the page
    res = requests.get(f"{base_url}{url}")

    # Print a status tracking message to the console showing which URL is currently being targeted
    print(f"Now Scraping{base_url}{url}")

    # Parse the raw HTML source text using Python's built-in parser manual and create a navigable DOM tree
    soup = BeautifulSoup(res.text, "html.parser")

    # Locate and extract every HTML element container that belongs to the class "quote"
    quotes = soup.find_all(class_="quote")

    # Iterate sequentially through each individual quote element container found on the current page
    for quote in quotes:

        # Extract specific text components, strip tags, and append them as a nested dictionary into the master list
        all_quotes.append({
            # Find the inner element with class "text" and extract only the raw quote characters
            "text": quote.find(class_="text").get_text(),

            # Find the inner element with class "author" and extract the text string of the author's name
            "author": quote.find(class_="author").get_text(),

            # Find the first anchor tag and extract the destination string value of its hidden "href" attribute
            "bio-link": quote.find("a")["href"]
        })

    # Locate the container structural element for the "Next Page" button layout
    next_btn = soup.find(_class="next")

    # If the next button container exists, extract its hyperlink path; if not, assign None to break the loop
    url = next_btn.find("a")["href"] if next_btn else None

    # Force Python to sit idle for exactly 2 seconds to mimic human browsing behavior and be polite to the server
    sleep(2)




Now Scrapinghttp://quotes.toscrape.com//page/1


/tmp/ipykernel_20632/2596734996.py:36: AttributeResemblesVariableWarning: '_class' is an unusual attribute name and is a common misspelling for 'class_'.

If you meant 'class_', change your code to use it, and this warning will go away.

If you really did mean to check the '_class' attribute, this warning is spurious and can be filtered. To make it go away, run this code before creating your BeautifulSoup object:

    from bs4 import AttributeResemblesVariableWarning
    import warnings

    warnings.filterwarnings("ignore", category=AttributeResemblesVariableWarning)

  next_btn = soup.find(_class="next")


In [3]:
# ==============================================================================
# SECTION 3: THE GAME INITIALIZATION BLOCK
# ==============================================================================

# Select exactly one random quote dictionary item out of the fully populated master list
quote = choice(all_quotes)

# Set the maximum limit counter tracking the user's available attempts
remaining_guesses = 4

# Print game greeting headers to the console screen
print("Here's a quote:  ")

# Access and display the secret quote text value stored inside the chosen dictionary
print(quote["text"])

# Initialize an empty string variable for the user's input to prepare for the conditional loop evaluation
guess = ''




Here's a quote:  
“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


In [4]:
# ==============================================================================
# SECTION 4: THE GAMEPLAY & CONDITIONAL HINT LOOP BLOCK
# ==============================================================================

# Loop while the user's lowercased answer does not match the lowercased target author AND they still have attempts left
while guess.lower() != quote["author"].lower() and remaining_guesses > 0:

    # Prompt the player to type an answer via input console and display their current life counter
    guess = input(
        f"Who said this quote? Guesses remaining {remaining_guesses}")

    # Check if the raw string input exactly matches the case-sensitive name string of the author
    if guess == quote["author"]:
        # Print a winning alert message to the console screen
        print("CONGRATULATIONS!!! YOU GOT IT RIGHT")
        # Terminate the loop structure immediately because the player won
        break

    # Deduct 1 attempt from the remaining life counter variable since the answer was incorrect
    remaining_guesses -= 1

    # HINT LEVEL 1: Triggered immediately when the user fails their first attempt and drops down to 3 lives
    if remaining_guesses == 3:
        # Construct the target path and execute an HTTP request to download the specific author's bio web page
        res = requests.get(f"{base_url}{quote['bio-link']}")
        # Parse the author's biography page HTML code into a searchable document object
        soup = BeautifulSoup(res.text, "html.parser")
        # Find the specific element containing the text for the author's historical date of birth
        birth_date = soup.find(class_="author-born-date").get_text()
        # Find the specific element containing the text for the author's historical location of birth
        birth_place = soup.find(class_="author-born-location").get_text()
        # Print a compiled hint string displaying the dynamically scraped birth info variables
        print(
            f"Here's a hint: The author was born on {birth_date}{birth_place}")

    # HINT LEVEL 2: Triggered when the user fails their second attempt and drops down to 2 lives
    elif remaining_guesses == 2:
        # Access index position 0 of the author's name string to extract the very first character of their first name
        print(
            f"Here's a hint: The author's first name starts with: {quote['author'][0]}")

    # HINT LEVEL 3: Triggered when the user fails their third attempt and drops down to their final life
    elif remaining_guesses == 1:
        # Cut the full name string into items at the space, select the second item (last name), and extract its first letter
        last_initial = quote["author"].split(" ")[1][0]
        # Print the extracted single character initial of the author's surname
        print(
            f"Here's a hint: The author's last name starts with: {last_initial}")

    # LOSS CONDITION: Triggered when the loop processing reaches an execution step where remaining_guesses equals 0
    else:
        # Print a game over message revealing the correct hidden target author string from the dictionary object
        print(
            f"Sorry, you ran out of guesses. The answer was {quote['author']}")

Who said this quote? Guesses remaining 4d
Here's a hint: The author was born on March 14, 1879in Ulm, Germany
Who said this quote? Guesses remaining 3d
Here's a hint: The author's first name starts with: A
Who said this quote? Guesses remaining 2d
Here's a hint: The author's last name starts with: E
Who said this quote? Guesses remaining 1d
Sorry, you ran out of guesses. The answer was Albert Einstein


In [5]:
import requests
from bs4 import BeautifulSoup

# Let's test the typo version first
base_url = "http://quotes.toscrape.com/"
url_typo = "/page/1"

res = requests.get(f"{base_url}{url_typo}")
soup = BeautifulSoup(res.text, "html.parser")

# 1. Test with the typo (_class)
next_btn_typo = soup.find(_class="next")
print("--- TESTING TYPO VERSION ---")
print(f"What the typo sees: {next_btn_typo}")

# 2. Test with the correct parameter (class_)
next_btn_correct = soup.find(class_="next")
print("\n--- TESTING CORRECT VERSION ---")
print(f"What the correct version finds:\n{next_btn_correct}")

--- TESTING TYPO VERSION ---
What the typo sees: None

--- TESTING CORRECT VERSION ---
What the correct version finds:
<li class="next">
<a href="/page/2/">Next <span aria-hidden="true">→</span></a>
</li>


/tmp/ipykernel_20632/1714731332.py:12: AttributeResemblesVariableWarning: '_class' is an unusual attribute name and is a common misspelling for 'class_'.

If you meant 'class_', change your code to use it, and this warning will go away.

If you really did mean to check the '_class' attribute, this warning is spurious and can be filtered. To make it go away, run this code before creating your BeautifulSoup object:

    from bs4 import AttributeResemblesVariableWarning
    import warnings

    warnings.filterwarnings("ignore", category=AttributeResemblesVariableWarning)

  next_btn_typo = soup.find(_class="next")
